<h1>Chapter 6 - Planning</h1>
<i>Autonomy for your `TinyAgent` through Native Tool Calling and Reasoning</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 6 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [16]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [17]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 3`

At the beginning of every chapter, we start by choosing the LLM that we want to use. In this notebook, we will explore how to enable autonomous behavior for LLMs that do not have native tool calling and reasoning behavior. As such, the model that we will be using throughout this chapter is Gemma 3, a model that does not have native tool calling capabilities.

In [1]:
from openai import OpenAI
from illustrated_agents.chapters.ch2 import LLM

# Ollama through OpenAI API
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma3:12b", client=client)

# Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", client=client)

# LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it", client=client)

# Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = LLM(model="gemini-2.5-flash", client=client)

## 2 - Adding **`Autonomy`**

In the previous chapter, we added the `Tools` module to your `TinyAgent`. In this chapter, we will cover how to give it more autonomy:

![../images/ch6.png](../images/ch6.png)

Throughout this notebook we will implement a module that allows for action sequencing, namely Reason and Act (ReAct). 

## 3 - Reason and Act (ReAct)

Autonomy, as covered in the book, is typically achieved through some method of task decomposition or planning. We are going to use the ReAct framework to give this autonomy to your `TinyAgent`. To do so, we are going to implement loops of:

* `THOUGHT` - A reasoning step about the current situation
* `ACTION` - An action to execute (e.g., a tool)
* `OBSERVATION` - A generated observation (typically the output of a tool)

Note that in this example, the `OBSERVATION` will be provided by us and not the LLM. This allows for a more simplified example as it would otherwise need quite a bit of code to add this observation. 

ReAct is mostly about prompting the LLM, so that's what we are going to do! The `ReAct` class consists of three functions:

* `system_prompt` - The ReAct prompt
* `parse_react` - Parse the LLM generated `THOUGHT` and `ACTION` so that they are separated for tool usage
* `format_observation` - Create the `OBSERVATION` based on the output of the tool

We instruct the `TinyAgent` to always use an `ACTION` and that if it does not need a tool, it can simply return it as:

* `"final_answer"` - The final answer as text that **stops** the `TinyAgent` from running


In [2]:
import re
from illustrated_agents.chapters.ch2 import Response

class ReAct:
    """ReAct module."""

    def __init__(self, max_steps: int = 10):
        """Initialize ReAct module.

        Arguments:
            max_steps: Maximum number of ReAct steps to perform.
        """
        self.max_steps = max_steps

    @property
    def prompt(self) -> str:
        return """
# ReAct (Reason and Act)

You are a ReAct agent that performs exactly ONE step per turn.

## ReAct Format

You use the following format for each step:

THOUGHT: [Your reasoning about what to do next]
ACTION:
{
    "tool": "a_tool_name",
    "kwargs": {"param": "value"},
}

An observation will be provided after each action. You do not generate the observation yourself.

## ReAct Completion

To provide the final answer to the task, use an action blob with "tool": "final_answer" tool. 
It is the only way to complete the task, else you will be stuck on a loop. 
So your final output should look like this:

ACTION:
{
    "tool": "final_answer",
    "kwargs": "insert your final answer here"
}

Use the `final_answer` tool when you are completely done with all subtasks and have the final answer ready.
You can also use `final_answer` to directly reply to a user's question without using any other tools.

"""

    def parse(self, response: Response) -> Response:
        """Parse a ReAct formatted response into THOUGHT and ACTION."""
        text = response.content

        # The patterns for each section
        patterns = {
            "THOUGHT": r"THOUGHT:\s*(.+?)(?=ACTION:|OBSERVATION:|$)",
            "ACTION": r"ACTION:\s*(.+?)(?=THOUGHT:|OBSERVATION:|$)",
        }

        # Extract each section using regex
        result = {}
        for key, pattern in patterns.items():
            match = re.search(pattern, text, re.DOTALL)
            result[key] = match.group(1).strip() if match else ""

        # Update Response and extract only the action
        response.content = result["ACTION"]
        response.reasoning = result["THOUGHT"]
        return response

There is a lot happening there, so let's go through each function step-by-step starting with the prompt. Since ReAct is mostly a prompting technique, the prompt in itself is the most important step. The prompt, which we crafted through careful trial and error, has several components that describe how your `TinyAgent` should behave. Below, we have annotated this function for you so it is clear what the highlights are. Note that we might update prompt slight, but the general idea should remain the same:


In [3]:
from illustrated_agents.chapters.ch6 import react_prompt_annotated; react_prompt_annotated

The `parse_react` function is needed to parse the strings that contain THOUGHT and ACTION into separate entities:

In [4]:
from illustrated_agents.chapters.ch6 import react_parse_annotated; react_parse_annotated

---

💡 **NOTE 1**: For execution, we only return the `ACTION` section since that's what we need to execute tools and provide answers.   
The THOUGHT section is useful for interpretability but not needed for execution. 

💡 **NOTE 2**: Use regular expressions (`re`) is definitely not the most stable way of doing this! What if the LLM makes a small mistake and uses "*THOUGH:*" instead of "*THOUGHT:*"? This will not capture and that is on purpose. We want to showcase the most minimal way of approaching this and the downsides of doing this. Likewise, we could construct a 100+ line function/class that is more robust but defeats the purpose of an educational example. That said, this downside also nicely demonstrates why native tool calling is more robust since the LLM has seen very specific tokens it can use to perform tool calling. This drastically reduces the error rate but makes an Agent a bit more of a blackbox. Don't worry, we're definitely also showing you how to do this in `chapter06_native_react.ipynb`.

---

## 4 - Updating `agent.py`

There are several changes needed to `TinyAgent` that requires changes throughout the class. In particular:

* `__init__` -- We need to add the prompt of the `ReAct` to the system prompt
* `run` -- Your `TinyAgent` now runs for a number of steps, defined by `ReAct`
* `_step` -- Now performs a single step of the `ReAct` loop (**THOUGHT**/**ACTION**/**OBSERVATION**)
* `_execute_action` -- A function to parse the tool call from the **ACTION** step.

In [5]:
from illustrated_agents.chapters.ch2 import Response, Trajectory
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import Tools


class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools, planner: ReAct):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner

        self.trajectory = Trajectory()

        # Build system prompt with all components
        system_prompt = "You are a helpful assistant.\n\n"
        system_prompt += self.planner.prompt
        system_prompt += self.tools.prompt
        self.memory.add("system", system_prompt)

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)
        self.trajectory.initialize(task)

        # *Autonomy* loop
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

    def _step(self) -> str | None:
        """Perform a single step."""
        # THOUGHT: Generate response and add to memory
        response = self.llm.generate(
            self.memory.get_messages(), tools=self.tools.schemas
        )
        self.memory.add(
            "assistant", response.content, tool_call=response.tool_call
        )

        # Tool parsing
        response = self.planner.parse(response)
        response = self.tools.parse(response)

        # ANSWER: Stopping mechanism
        if self.tools.is_done(response):
            self.trajectory.add(response)
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """Execute a tool action."""

        # ACTION: execute tools
        result = self.tools.execute(response)

        # OBSERVATION: add tool results to memory and display
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)
        self.trajectory.add(response, observation)

        return None


Although we can show the diff like we did before, this time there are so many changes! Instead, let's go through each function and describe how they are used, starting with the `__init__`:

In [6]:
from illustrated_agents.chapters.ch6 import tinyagent_init_annotated; tinyagent_init_annotated

Next, let's explore the `run` function where two interesting actions take place:

In [7]:
from illustrated_agents.chapters.ch6 import tinyagent_run_annotated; tinyagent_run_annotated

Next, the `_step` is where much of the processing happens and now shows an interesting structure:

In [8]:
from illustrated_agents.chapters.ch6 import tinyagent_step_annotated; tinyagent_step_annotated

Finally, the `_execute_action` processes the **ACTION** step:

In [9]:
from illustrated_agents.chapters.ch6 import tinyagent_action_annotated; tinyagent_action_annotated

We can also show it all the changes at once. Compared to previous chapters, the difference is a bit bigger as a result of the `for` loop:

In [10]:
from illustrated_agents.chapters.ch6 import tinyagents_diff; tinyagents_diff

Now that we went through all steps, let's start creating your ReAct-based `TinyAgent` and explore whether it can actually do things autonomously!

## 6 - Running The Autonomous `TinyAgent`

In [65]:
from illustrated_agents.toolbox import add, multiply, subtract

# Register tools
tools = Tools()
tools.add_tool("add", add, "add(a: str, b: str)")
tools.add_tool("subtract", subtract, "subtract(a: str, b: str)")
tools.add_tool("multiply", multiply, "multiply(a: str, b: str)")

# Memory
memory = Memory()

# ReAct
react = ReAct(max_steps=10)

# Create agent
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react)

# Multi-step task with reasoning
agent.run("What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?")

'42.0'

Amazing! If you used **Gemma 3 12B** then it should have run correctly. It performed all tasks in sequence and gave us back the final reply. Let's explore how many steps it took:

In [25]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

In [68]:
from rich import print

print(agent.trajectory.runs)

[
    {
        'query': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?',
        'steps': [
            Step(
                thought='First, I need to add 4.6 and 6.685. Then, I need to multiply the result by 4. Finally, I 
need to subtract 3.14 from that final result.',
                action={'tool': 'add', 'kwargs': {'a': '4.6', 'b': '6.685'}},
                observation='OBSERVATION: 11.285',
                answer=None,
                metadata=None
            ),
            Step(
                thought="Now that I've added 4.6 and 6.685, which resulted in 11.285, I need to multiply this by 
4.",
                action={'tool': 'multiply', 'kwargs': {'a': '11.285', 'b': '4'}},
                observation='OBSERVATION: 45.14',
                answer=None,
                metadata=None
            ),
            Step(
                thought='I have now multiplied 11.285 by 4, obtaining 45.14. The final step is to subtract 3.14 
from this result.',
                action={'tool': 'subtract', 'kwargs': {'a': '45.14', 'b': '3.14'}},
                observation='OBSERVATION: 42.0',
                answer=None,
                metadata=None
            ),
            Step(
                thought="I've completed all the necessary calculations. I added 4.6 and 6.685, multiplied the 
result by 4, and then subtracted 3.14. The final answer is 42.0.",
                action=None,
                observation=None,
                answer='42.0',
                metadata=None
            )
        ]
    }
]

In our example, 3 tools were called in sequence. Then, your `TinyAgent` created a final message with the result. As always, let's end with exploring the Agent's memory:

In [69]:
from rich import print
print(agent.memory.messages)

[
    {
        'role': 'system',
        'content': 'You are a helpful assistant.\n\n\n# ReAct (Reason and Act)\n\nYou are a ReAct agent that 
performs exactly ONE step per turn.\n\n## ReAct Format\n\nYou use the following format for each step:\n\nTHOUGHT: 
[Your reasoning about what to do next]\nACTION:\n{\n    "tool": "a_tool_name",\n    "kwargs": {"param": 
"value"},\n}\n\nAn observation will be provided after each action. You do not generate the observation 
yourself.\n\n## ReAct Completion\n\nTo provide the final answer to the task, use an action blob with "tool": 
"final_answer" tool. \nIt is the only way to complete the task, else you will be stuck on a loop. \nSo your final 
output should look like this:\n\nACTION:\n{\n    "tool": "final_answer",\n    "kwargs": "insert your final answer 
here"\n}\n\nUse the `final_answer` tool when you are completely done with all subtasks and have the final answer 
ready.\nYou can also use `final_answer` to directly reply to a user\'s question without using any other 
tools.\n\n\n# Tools\n\nIf needed, you can only use the following tools to assist you in completing tasks:\n\n`add`:
add(a: str, b: str)\n`subtract`: subtract(a: str, b: str)\n`multiply`: multiply(a: str, b: str)\n\nTo use a tool, 
respond with JSON: {"tool": "name", "kwargs": {"param": "value"}}\n'
    },
    {'role': 'user', 'content': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: First, I need to add 4.6 and 6.685. Then, I need to multiply the result by 4. Finally,
I need to subtract 3.14 from that final result.\nACTION:\n{\n    "tool": "add",\n    "kwargs": {"a": "4.6", "b": 
"6.685"}\n}\n'
    },
    {'role': 'user', 'content': 'OBSERVATION: 11.285'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: Now that I\'ve added 4.6 and 6.685, which resulted in 11.285, I need to multiply this 
by 4.\nACTION:\n{\n    "tool": "multiply",\n    "kwargs": {"a": "11.285", "b": "4"}\n}'
    },
    {'role': 'user', 'content': 'OBSERVATION: 45.14'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: I have now multiplied 11.285 by 4, obtaining 45.14. The final step is to subtract 3.14
from this result.\nACTION:\n{\n    "tool": "subtract",\n    "kwargs": {"a": "45.14", "b": "3.14"}\n}'
    },
    {'role': 'user', 'content': 'OBSERVATION: 42.0'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: I\'ve completed all the necessary calculations. I added 4.6 and 6.685, multiplied the 
result by 4, and then subtracted 3.14. The final answer is 42.0.\nACTION:\n{\n    "tool": "final_answer",\n    
"kwargs": "42.0"\n}'
    }
]

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered what makes your `TinyAgent` autonomous... a for-loop! This for-loop, driven by the Reason and Act (ReAct) framework is quite capable. As we covered in the book, ReAct is not something we always will do explicitly because newer models tend to be trained on that already. Going through the steps of it helps you understand what the LLM is doing under the hood that makes this all possible. 

In [70]:
from illustrated_agents.chapters.ch6 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py      ← Updated (Autonomy with a for-loop and ReAct planner.)                                       │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py   ← New (Added the ReAct (Reason and Act) framework)                                            │
│ ├── toolbox.py                                                                                                  │
│ ├── tools.py                                                                                                    │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

...